This notebook is for running Unet on UMN clinical data 

Masks were created by Nora, G, Vera, and Hayoung

In [30]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import skimage.morphology as mo
from skimage import io, color #Scikit-Image
from PIL import Image # Pillow
import cv2
import os
import random
import torch # Will work on using PyTorch here later
from torch.utils.data  import Dataset, DataLoader
from torchvision import transforms
import torchvision.transforms.functional as TF
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt
import pandas as pd
import torch

print(torch.cuda.is_available())

if torch.backends.mps.is_available():
    mps_device = torch.device("mps")
    x = torch.ones(1, device=mps_device)
    print (x)
else:
    print ("MPS device not found.")

False
tensor([1.], device='mps:0')


In [32]:
# Init transform functions
tx_X = transforms.Compose([transforms.Resize((256, 256)),
                           transforms.ToTensor(),
                           transforms.Normalize((0.5,), (0.5,))])
tx_Y = transforms.Compose([transforms.Resize((256, 256)),
                           transforms.ToTensor(),  ################ no need to normalize the mask
                           # transforms.Normalize((0.5,), (0.5,))
                          ])

if __name__ == "__main__":
    train_data = Muscle(train=True, transformX=tx_X, transformY=tx_Y)
    train_loader = DataLoader(dataset=train_data, batch_size=8, shuffle=True, num_workers=2)
    validation_data = Muscle(train = False, transformX = tx_X, transformY = tx_Y)

FileNotFoundError: [Errno 2] No such file or directory: '/Users/taliacho/Downloads/Ranger/Bryan-Ranger/clinical_data/UM_masked/UMN_train.csv'

In [23]:
# Dataloaders
batch_size=8
train_loader = DataLoader(dataset = train_data, batch_size = 8, shuffle = True, num_workers = 2)
validation_loader = DataLoader(dataset = validation_data, batch_size = 8, shuffle = True, num_workers = 2)
print(len(train_loader)) #len(train_loader)*batch_size = total number of images in training set (50*8 = 400), hayo: now 30?
print(len(validation_loader)) #len(validation_loader)*batch_size = total number of images in validation set(13*8 = 104 ~ 100)

#hayo the validation_loader is now 8 instead of 13
print(len(validation_loader)*batch_size)

30
8
64


In [24]:
# The following functions will return numpy array from the transformed tensors which were
# obtained from our train_loader. Plot them and see if they are intact
def im_converterX(tensor):
  image = tensor.clone().detach().numpy() # make copy of tensor and converting it to numpy
                                              # as we will need original later
  image = image.transpose(1,2,0) # swapping axes making (1, 28, 28) image to a (28, 28, 1)
  print("image shape is ",image.shape)
  image = image * np.array((0.5, 0.5, 0.5)) + np.array((0.5, 0.5, 0.5)) # unnormalizing the image
                                              # this also outputs (28, 28, 3) which seems important for plt.imshow
  image = image.clip(0, 1) # to make sure final values are in range 0 to 1 as .ToTensor outputed
  return image

def im_converterY(tensor):
  image = tensor.clone().detach().numpy()
  image = image.transpose(1,2,0)
  print("image shape is ",image.shape)
  image = image * np.array((1, 1, 1))
  image = image.clip(0, 1)
  return image

In [26]:
## Here we loop through our train_loader and see the images
fig = plt.figure(figsize = (15,6))

for ith_batch, sample_batched in enumerate(train_loader):
    print(ith_batch, sample_batched['image'].size(), sample_batched['mask'].size())

    for index in range(2):
        ax = fig.add_subplot(2, 2 , index + 1)  # subplot index starts from 1
        plt.imshow(im_converterX(sample_batched['image'][index]))
        ax = fig.add_subplot(2, 2, index + 3)
        plt.imshow(im_converterY(sample_batched['mask'][index]))
    break

Traceback (most recent call last):
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "<string>", line 1, in <module>
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 120, in spawn_main
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 120, in spawn_main
        exitcode = _main(fd, parent_sentinel)exitcode = _main(fd, parent_sentinel)

                              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 130, in _main
  File "/Users/taliacho/anaconda3/lib/python3.11/multiprocessing/spawn.py", line 130, in _main
    self = reduction.pickle.load(from_parent)
    self = reduction.pickle.load(from_parent)
           ^^^^^^^^^^^ ^ ^ ^ ^ ^ ^ ^ ^ ^ ^ ^^^^^^^^^^^^^^^^^^^^^^^^^^
^^^^^AttributeError^^: ^Can't get attribute 'Muscle' on <module '__main__' (built-in)>^
^^^^^^^^^^^^
AttributeError: Can't get attri

RuntimeError: DataLoader worker (pid(s) 83131) exited unexpectedly

<Figure size 1500x600 with 0 Axes>